# Data Processing

Downloaded the catalytically active ORFs (HMM validated) 300M in total from here [s3://petadex/logan/petadex.catalytic_orfs.v1.1.fa](s3://petadex/logan/petadex.catalytic_orfs.v1.1.fa)

In [ ]:
!seqkit stats petadex.catalytic_orfs.v1.1.fa

```
file                            format  type        num_seqs          sum_len  min_len  avg_len  max_len
petadex.catalytic_orfs.v1.1.fa  FASTA   Protein  307,155,746  102,929,980,061       25    335.1   59,275

```

Downloaded table of 90 percent centroids from SQL and saved them here s3://petabite/petadex/90pid_enzymes_clusters.csv

In [ ]:
cat 90pid_enzymes_clusters.csv | cut -f 2 -d ',' > 90_orfs.txt

In [ ]:
wc -l 90_orfs_sorted.txt

18172960

In [ ]:
!LC_ALL=C sort --parallel=16 -S 24G 90_orfs.txt -o 90_orfs_sorted.txt
!LC_ALL=C sort --parallel=16 -S 24G orf_id.txt -o orf_id_sorted.txt
!LC_ALL=C comm -23 90_orfs_sorted.txt orf_id_sorted.txt | wc -l

0

In [ ]:
seqkit stats 90pid_catalytic_orfs.fa

```
file                     format  type       num_seqs        sum_len  min_len  avg_len  max_len
90pid_catalytic_orfs.fa  FASTA   Protein  18,172,960  4,998,232,825       29      275   59,275
```

In [ ]:
seqkit fx2tab 90pid_catalytic_orfs.fa > 90pid_catalytic_orfs.tsv
tr '|' '\t' < 90pid_catalytic_orfs.tsv > 90pid_catalytic_orfs_1.tsv
mv 90pid_catalytic_orfs_1.tsv 90pid_catalytic_orfs.tsv
sed -i 's/\t$//' 90pid_catalytic_orfs.tsv

In [ ]:
printf 'ORF ID\tGenbank Acc\tLibrary\tContig\tORF Start\tORF End\tORF Type\tSequence\n' | cat - 90pid_catalytic_orfs.tsv > tmp && mv tmp 90pid_catalytic_orfs.tsv

In [1]:
import pandas as pd

In [2]:
centroids = pd.read_csv('../data/90pid_catalytic_orfs.tsv', sep='\t')

/tmp/ipykernel_1477038/1340131322.py:1: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  centroids = pd.read_csv('../data/90pid_catalytic_orfs.tsv', sep='\t')


In [3]:
centroids

,ORF ID,Genbank Acc,Library,Contig,ORF Start,ORF End,ORF Type,Sequence
0,116438891,NaN,ERR1811635,211282.0,2.0,644.0,1.0,ARGARAQAERIDAAVARGEDPGPLAGVPFGVKDVEKAAGLPATRGS...
1,118703989,NaN,ERR1879305,79477.0,0.0,1317.0,1.0,TGLAPIPRLSEVLEQYRRCERTPSDVVEECLARIEALEPQLKAWVL...
2,119313590,NaN,ERR1883884,1392981.0,0.0,1833.0,0.0,LRAPRRGACGARRGRPRTSRLLVSGHEHHPRDGAVRGPAGAASPVG...
3,119842227,NaN,ERR1906297,504.0,176.0,1652.0,0.0,RWRAWSTNSRCPTKPNRPVSMQLDSVITPEGDWRPASEIAQAVATQ...
4,119849203,NaN,ERR1906324,553.0,3582.0,5256.0,0.0,CRYKGKPNRHTMKMPARRDPPLAAVAMSAEAAIMLKLETAPMSDII...
...,...,...,...,...,...,...,...,...
18172955,1327309062,NaN,SRR8863204,588325.0,0.0,243.0,1.0,IDPTTGKVVAGVRNADGTLADARRLLRGGASIAVPRAGENTTVAVV...
18172956,1327156605,NaN,SRR8863179,18057962.0,2.0,194.0,3.0,VRSNTTLAVVATNARLTKVEATKLAQQASTGMVRAISPVNTMSDGD...
18172957,1327134933,NaN,SRR8863179,4976090.0,1.0,397.0,1.0,GPAKVPIVAGAILFDLGIGDAAARPDAAAGYAACQTASEGPVEEGS...
18172958,1327194666,NaN,SRR8863180,833178.0,81.0,1269.0,0.0,PGRGPLRGPPDRCHATRSHRRARRDRAVLRDPRTPLFPPAPGGPVM...


## Train / Val / Test splits → Hugging Face (`petadex`)

The `centroids` dataframe above holds **18,172,960** catalytic ORFs (90% identity centroids — already dereplicated, so a plain random split has no cluster leakage). Below we build a 🤗 `Dataset` from it, make a **98 / 1 / 1** train / validation / test split, and push it (all columns) to the **private** `petadex/catalytic-orfs-90pid` dataset for ESM-C LoRA/PEFT fine-tuning.

In [4]:
from datasets import Dataset, DatasetDict

SEED = 42

# Build a Dataset straight from the dataframe loaded above (keeps all 8 columns).
ds = Dataset.from_pandas(centroids, preserve_index=False)

# 98 / 1 / 1 split: hold out 2%, then halve it into validation + test (~182k each).
holdout  = ds.train_test_split(test_size=0.02, seed=SEED)
val_test = holdout["test"].train_test_split(test_size=0.5, seed=SEED)

splits = DatasetDict({
    "train":      holdout["train"],
    "validation": val_test["train"],
    "test":       val_test["test"],
})

/home/purav/petabite/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
splits

DatasetDict({
    train: Dataset({
        features: ['ORF ID', 'Genbank Acc', 'Library', 'Contig', 'ORF Start', 'ORF End', 'ORF Type', 'Sequence'],
        num_rows: 17809500
    })
    validation: Dataset({
        features: ['ORF ID', 'Genbank Acc', 'Library', 'Contig', 'ORF Start', 'ORF End', 'ORF Type', 'Sequence'],
        num_rows: 181730
    })
    test: Dataset({
        features: ['ORF ID', 'Genbank Acc', 'Library', 'Contig', 'ORF Start', 'ORF End', 'ORF Type', 'Sequence'],
        num_rows: 181730
    })
})

In [6]:
# Uploads parquet shards per split and creates the repo if it doesn't exist (~5 GB, takes a while).
splits.push_to_hub("petadex/catalytic-orfs-90pid", private=False)

Creating parquet from Arrow format: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [01:32<00:00, 18.41s/ba]
Processing Files (0 / 0): |                                                                                                                                                               |  0.00B /  0.00B            
Processing Files (0 / 1):   0%|▏                                                                                                                                                          |  524kB /  412MB, 51.6kB/s  
Processing Files (0 / 1):   2%|██▎                                                                                                                                                        | 6.29MB /  412MB,  617kB/s  
Processing Files (0 / 1):   5%|███████▎                                                                                                 

CommitInfo(commit_url='https://huggingface.co/datasets/petadex/catalytic-orfs-90pid/commit/1578cb113d6b08489136b6dab98c667411d7ec19', commit_message='Upload dataset', commit_description='', oid='1578cb113d6b08489136b6dab98c667411d7ec19', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/petadex/catalytic-orfs-90pid', endpoint='https://huggingface.co', repo_type='dataset', repo_id='petadex/catalytic-orfs-90pid'), pr_revision=None, pr_num=None)

### Load it back / next steps

```python
from datasets import load_dataset

ds = load_dataset("petadex/catalytic-orfs-90pid")
```